# CourtListener → DataFrame (HCDE 530)

Uses the **local CourtListener CSV export** (no API token or live calls).

**File:** `MiniProject1/data/courtlistener.csv` or `week 6/Week6_files/courtlistener_week5_repull_40k.csv` (same data).

The export has **W.D. Wash.** (`court_id:wawd`) and **King County (Wash. Ct. App.)** (`washctapp`) rows. Party names are parsed from `case` when it looks like *Party A v. Party B*.

Install: `python3 -m pip install pandas` (same interpreter as this notebook).

1. **Sample opinions** — 25 rows with parseable party names (replaces a Miranda API search on this slice).
2. **Seattle jurisdiction** — 20 federal `wawd` rows with **jurisdiction** = **Seattle jurisdiction**.
3. **Most recent judgments** — 20 newest `washctapp` rows by `date_of_decision`.


In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display


def _ensure_loader() -> None:
    here = Path.cwd().resolve()
    for base in [here, *here.parents]:
        w6 = base / "week 6" / "Week6_files"
        if (w6 / "courtlistener_csv_loader.py").is_file():
            p = str(w6)
            if p not in sys.path:
                sys.path.insert(0, p)
            return
    raise FileNotFoundError("Add week 6/Week6_files/courtlistener_csv_loader.py to the repo.")


_ensure_loader()
from courtlistener_csv_loader import (
    SEATTLE_JURISDICTION_LABEL,
    load_courtlistener_csv,
    slice_king_county_recent,
    slice_sample_opinions,
    slice_seattle_federal,
    split_parties,
)

master = load_courtlistener_csv()
print("Courts in CSV:", master["court_id"].value_counts().to_dict())

sample = slice_sample_opinions(master, n=25)
rows: list[dict[str, object]] = []
for _, r in sample.iterrows():
    plaintiff, defendant = split_parties(str(r["case"]))
    rows.append(
        {
            "case": r["case"],
            "plaintiff": plaintiff or pd.NA,
            "defendant": defendant or pd.NA,
            "judge": r["judge"] or pd.NA,
            "date of decision": r["date_of_decision"],
        }
    )
df = pd.DataFrame(rows)
display(df)

display(
    Markdown(
        "## Judgments classified by Judge names and jurisdiction\n\n"
        "Federal Seattle-area opinions from the CSV (`court_id` **wawd**). "
        "Column **jurisdiction** is **Seattle jurisdiction** for each row."
    )
)

wawd = slice_seattle_federal(master, n=20)
rows_jurisdiction: list[dict[str, object]] = []
for _, r in wawd.iterrows():
    rows_jurisdiction.append(
        {
            "judge": r["judge"] or pd.NA,
            "jurisdiction": SEATTLE_JURISDICTION_LABEL,
            "court": r["court"] or pd.NA,
            "court_id": r["court_id"],
            "case": r["case"],
            "date of decision": r["date_of_decision"],
        }
    )
df_jurisdiction = pd.DataFrame(rows_jurisdiction)
display(df_jurisdiction)

display(
    Markdown(
        "## Most recent judgments (King County Court of Appeals)\n\n"
        "From the local export: **`washctapp`** rows (dataset label King County), "
        "sorted by **`date_of_decision`** descending — top **20**."
    )
)

kc = slice_king_county_recent(master, n=20)
rows_recent: list[dict[str, object]] = []
for _, r in kc.iterrows():
    rows_recent.append(
        {
            "case": r["case"],
            "judge": r["judge"] or pd.NA,
            "court": r["court"] or pd.NA,
            "court_id": r["court_id"],
            "date filed": r["date_of_decision"],
        }
    )
df_recent = pd.DataFrame(rows_recent)
df_recent


Loaded 39,040 rows from /Users/rnr6/Documents/HCDE/hcde530/MiniProject1/data/courtlistener.csv
Courts in CSV: {'washctapp': 36000, 'wawd': 3040}


,case,plaintiff,defendant,judge,date of decision
0,"State Of Washington, V. Justin R. Smith","State Of Washington,",Justin R. Smith,NaN,2025-07-21
1,"Chen Wang And Liyin Xue, V. Dongmei Huang","Chen Wang And Liyin Xue,",Dongmei Huang,NaN,2025-08-25
2,"Erwin Chappel, Respondent/cr-appellants V. Dou...","Erwin Chappel, Respondent/cr-appellants","Douglas Johnson, Appellant/cr-respondents",NaN,2025-09-15
3,"Alaska Airlines, V. Hillary Spanjer","Alaska Airlines,",Hillary Spanjer,NaN,2025-09-15
4,"Samantha Snodderly, V. Bradley Shockey","Samantha Snodderly,",Bradley Shockey,NaN,2025-09-22
5,"State Of Washington, V. Kimcha Chhim","State Of Washington,",Kimcha Chhim,NaN,2025-08-19
6,"Clifton A. Little Ii Et Ano, V. Hardie-tynes C...","Clifton A. Little Ii Et Ano,",Hardie-tynes Co. Inc.,NaN,2025-08-25
7,State of Washington v. Christopher Floe,State of Washington,Christopher Floe,NaN,2025-07-29
8,"Lisa Earl, V. City Of Tacoma, Scott Campbell","Lisa Earl,","City Of Tacoma, Scott Campbell",NaN,2025-06-17
9,"State Of Washington, V. Karen K. Peterson","State Of Washington,",Karen K. Peterson,NaN,2025-08-04


## Judgments classified by Judge names and jurisdiction

Federal Seattle-area opinions from the CSV (`court_id` **wawd**). Column **jurisdiction** is **Seattle jurisdiction** for each row.

,judge,jurisdiction,court,court_id,case,date of decision
0,NaN,Seattle jurisdiction,"District Court, W.D. Washington",wawd,"Simmons v. Safeway, Inc.",2019-08-01
1,Settle,Seattle jurisdiction,"District Court, W.D. Washington",wawd,State v. Franciscan Health Sys.,2019-03-01
2,Leighton,Seattle jurisdiction,"District Court, W.D. Washington",wawd,"Animal Legal Def. Fund v. Olympic Game Farm, Inc.",2019-05-21
3,Jones,Seattle jurisdiction,"District Court, W.D. Washington",wawd,"Beane v. RPW Legal Servs., PLLC",2019-05-06
4,Lasnik,Seattle jurisdiction,"District Court, W.D. Washington",wawd,Galvez v. Cuccinelli,2019-07-17
5,Leighton,Seattle jurisdiction,"District Court, W.D. Washington",wawd,Mitchell v. Atkins,2019-05-20
6,Robart,Seattle jurisdiction,"District Court, W.D. Washington",wawd,Calderon-Rodriguez v. Wilcox,2019-02-06
7,Pechman,Seattle jurisdiction,"District Court, W.D. Washington",wawd,Gallupe v. Sedgwick Claims Mgmt. Servs. Inc.,2019-02-14
8,Lasnik,Seattle jurisdiction,"District Court, W.D. Washington",wawd,"United Statesi Ins. Servs. Nat'l, Inc. v. Ogden",2019-03-06
9,Leighton,Seattle jurisdiction,"District Court, W.D. Washington",wawd,Marine Carpenters Pension Fund v. Puglia Marin...,2019-04-10


## Most recent judgments (King County Court of Appeals)

From the local export: **`washctapp`** rows (dataset label King County), sorted by **`date_of_decision`** descending — top **20**.

,case,judge,court,court_id,date filed
0,"Samantha Snodderly, V. Bradley Shockey",NaN,Court of Appeals of Washington,washctapp,2025-09-22
1,"Samantha Snodderly, V. Bradley Shockey",NaN,Court of Appeals of Washington,washctapp,2025-09-22
2,"Samantha Snodderly, V. Bradley Shockey",NaN,Court of Appeals of Washington,washctapp,2025-09-22
3,"Samantha Snodderly, V. Bradley Shockey",NaN,Court of Appeals of Washington,washctapp,2025-09-22
4,"Samantha Snodderly, V. Bradley Shockey",NaN,Court of Appeals of Washington,washctapp,2025-09-22
5,"Samantha Snodderly, V. Bradley Shockey",NaN,Court of Appeals of Washington,washctapp,2025-09-22
6,"Samantha Snodderly, V. Bradley Shockey",NaN,Court of Appeals of Washington,washctapp,2025-09-22
7,"Samantha Snodderly, V. Bradley Shockey",NaN,Court of Appeals of Washington,washctapp,2025-09-22
8,"Samantha Snodderly, V. Bradley Shockey",NaN,Court of Appeals of Washington,washctapp,2025-09-22
9,"Samantha Snodderly, V. Bradley Shockey",NaN,Court of Appeals of Washington,washctapp,2025-09-22


### Notes

- **Data source**: local CSV only — no `COURTLISTENER_API_TOKEN` required to run this notebook.
- **Miranda search**: the bundled export is W.D. Wash. + King County appellate stacks; the first table uses **25 sample opinions** with *v.* party names instead of a live Miranda query.
- **King County**: newest 20 rows from `washctapp` in the export (dataset **King County (Wash. Ct. App.)**).
